# 🕵️ Case file 03 — The confident hallucination

> **Mission briefing.** The day gets worse. A Gold member was promised baggage they are not entitled to, and the answer sounded completely sure of itself. A confident wrong answer like this is often called a *hallucination*. It does not always come from where you expect. Open the activity logs, find where the wrong number came from, and make the assistant stick to the policy.

Each run leaves an **activity log** (Langfuse calls it a *trace*), made of **steps** (*spans*): the search step that found a policy, and the answer step (*generation*) that shows the exact instructions the assistant received. The notebook does not show those instructions. Only the activity log does.

## Case objective
Trace a wrong answer back to its source (the instructions, or the policy data) and correct it at that source.

## What to expect
Three levels. Each run prints the answer and a link to its activity log. The cause is never printed: you find it in the activity log.

## Bad practice
Fixing the wording of an answer without finding out where the wrong fact came from.

## Good practice
Compare the instructions, the policy that was found, and the answer, then correct whichever one is wrong.

## Why the good practice works
The assistant can only combine what it was given. When the activity log shows all three side by side, the contradiction is visible.

## Deliverable
A claim code for each level you solve, and for your demo, a before/after pair of activity-log links with the instruction you changed.

## 🎮 How the game works

- Put your team name in the next cell and run it. Use **the same name in every notebook**.
- Each level asks you to find something in the activity logs. Type your answer in `check_flag(...)` and run it. Wrong guesses cost nothing.
- Correct answers give a **claim code**. Paste it into the game master's form.
- Stuck? `hint("3-1")` shows a hint. Each hint costs 25% of that level's points (two hints at most).
- Points: Level 1 = 100, Level 2 = 200, Level 3 = 300.

In [1]:
from experiment_utils import (FAITHFUL_INSTRUCTIONS, check_flag, close_case, hint, run_hallucination,
                              run_rule_check, set_team, show_hallucination_results, show_rule_results)

TEAM = "your-team-name"  # ← your team name, the same in every notebook
set_team(TEAM)

⚠️  TEAM is still 'your-team-name'. Claim codes will not score: change it and rerun this cell.


'your-team-name'

## Level 1 — The generous promise (100 points)

A Gold member asks how many checked bags their fare includes. The assistant runs with its current instructions, which this notebook does not show you.

**Find:** the number of checked bags the assistant's *instructions* promise. Then write your own instructions in `my_prompt` and rerun to correct the answer.

In [2]:
broken = run_hallucination(None, "broken")
show_hallucination_results([broken])


level 1 · broken
  answer: As a Gold member, you’re entitled to **two checked bags**.

The retrieved policy says the Gold fare includes one 23 kg checked bag and a second normally costs 60 EUR—but Gold members are always entitled to two checked bags, so the second is included for you.
  activity log: http://localhost:3000/project/langfuse-observability/traces/d242f4a1584614cb3dec0ca254d3a5ce


In [3]:
check_flag("3-1", "type your answer here")
# hint("3-1")

❌ Not quite. Try again, or run hint("3-1").


In [4]:
my_prompt = "You are Skyline Travel's assistant."  # ← write instructions that keep the answer faithful to the policy
fixed = run_hallucination(my_prompt, "fixed")
show_hallucination_results([broken, fixed])


level 1 · broken
  answer: As a Gold member, you’re entitled to **two checked bags**.

The retrieved policy says the Gold fare includes one 23 kg checked bag and a second normally costs 60 EUR—but Gold members are always entitled to two checked bags, so the second is included for you.
  activity log: http://localhost:3000/project/langfuse-observability/traces/d242f4a1584614cb3dec0ca254d3a5ce

level 1 · fixed
  answer: Your Gold fare includes **one checked bag up to 23 kg**. A second checked bag is not included and costs **60 EUR**.
  activity log: http://localhost:3000/project/langfuse-observability/traces/7518b5a793b639d09bc968c72cf00f3a


## Level 2 — Faithful to the wrong page (200 points)

The instructions are now good: *follow the policy documents*. A new run still gives a wrong answer. The assistant did exactly what it was told. So what was it given?

**Find:** the id of the policy that caused the wrong answer. Then set `active_only = True` and rerun the second cell, so the search only uses policies that are still valid.

In [5]:
stale = run_hallucination(FAITHFUL_INSTRUCTIONS, "stale", level=2, active_only=False)
show_hallucination_results([stale])


level 2 · stale
  answer: Your Gold fare includes **two 23 kg checked bags at no extra charge**.
  activity log: http://localhost:3000/project/langfuse-observability/traces/afd5c2f42c5af97715346d7758a481fa


In [6]:
check_flag("3-2", "type your answer here")
# hint("3-2")

❌ Not quite. Try again, or run hint("3-2").


In [7]:
active_only = False  # ← change to True and rerun
retry = run_hallucination(FAITHFUL_INSTRUCTIONS, f"active-only-{active_only}".lower(), level=2, active_only=active_only)
show_hallucination_results([stale, retry])


level 2 · stale
  answer: Your Gold fare includes **two 23 kg checked bags at no extra charge**.
  activity log: http://localhost:3000/project/langfuse-observability/traces/afd5c2f42c5af97715346d7758a481fa

level 2 · active-only-false
  answer: Your Gold fare includes **two 23 kg checked bags at no extra charge**.
  activity log: http://localhost:3000/project/langfuse-observability/traces/4ed4caa280e038ea9493fea2fca0d64a


## Level 3 — Write the rule (300 points)

Real assistants often run on base instructions you cannot edit. Here, the old Level 1 instructions are **always sent first**, and your `my_prompt` is added after them. The search also hands the assistant **every matching policy, current and retired**, each marked with its status.

Three Gold-member questions follow: baggage allowance, second-bag price and free cancellation. Your instructions must make **all three** answers match the current policy. Edit `my_prompt` (in the Level 1 cell or here), then run the check. Each question has its own step in the activity log, so you can see what the assistant was given and which answer fails. When all three match, the notebook prints a flag.

In [8]:
rules = run_rule_check(my_prompt)
show_rule_results(rules)


level 3 · 1 of 3 answers match the policy
  ❌ How many checked bags are included in my Gold fare?
     As a Gold member, you’re entitled to **two checked bags** (23 kg each).  

Note: the active fare rule lists one included bag with a second for 60 EUR, but your Gold status entitles you to two checked bags.
  ❌ What does a second checked bag cost on my Gold fare?
     As a Gold member, your second checked bag is **included at no extra charge (0 EUR)** — Gold members are always entitled to two checked bags.
  ✅ Until when can I cancel my Gold ticket free of charge?
     You can cancel your Gold ticket free of charge up to **24 hours before departure**.
  activity log: http://localhost:3000/project/langfuse-observability/traces/1d0abf19151b9001606099577b0c40a0


In [9]:
check_flag("3-3", "type your answer here")
# hint("3-3")

❌ Not quite. Try again, or run hint("3-3").


## 🏁 Wrap-up

Run the last cell to close the case and send the activity logs to Langfuse.

In [10]:
close_case("activity-3", [broken, fixed, stale, retry, rules])


case activity-3 closed — session activity-3-69862720
level 1 · broken: http://localhost:3000/project/langfuse-observability/traces/d242f4a1584614cb3dec0ca254d3a5ce
level 1 · fixed: http://localhost:3000/project/langfuse-observability/traces/7518b5a793b639d09bc968c72cf00f3a
level 2 · stale: http://localhost:3000/project/langfuse-observability/traces/afd5c2f42c5af97715346d7758a481fa
level 2 · active-only-false: http://localhost:3000/project/langfuse-observability/traces/4ed4caa280e038ea9493fea2fca0d64a
level 3 · rule check, 1 of 3 match the policy: http://localhost:3000/project/langfuse-observability/traces/1d0abf19151b9001606099577b0c40a0
activity logs flushed to Langfuse
